## Bias, Fairness, and Accountability

---

### **0. Notebook Overview and Learning Goals**

In this notebook we begin our exploration of **bias, fairness, and accountability** in modern machine learning and deep learning systems. Until now, across Projects 1–11, we focused on *how models learn*, *how they generalize*, and *how we adapt them* using techniques like transfer learning and fine-tuning. In this project we shift our perspective: instead of asking *what the model can do*, we start asking *what the model should do*.

Deep learning systems do not learn in isolation. They learn from data created by us, shaped by our societies, and influenced by the assumptions and behaviours of different groups. When the data is unbalanced or reflects harmful stereotypes, the model can absorb those patterns. When the evaluation setup is incomplete, the model may appear accurate but behave unfairly in real-world situations. And when deployment decisions are made without thinking about accountability, people can be negatively affected.

Our goal in this notebook is to build an intuitive and structured understanding of how bias arises in machine learning systems and why fairness is not just a mathematical constraint but a foundational requirement for trustworthy AI.

To guide us, we focus on three questions:

1. **What kinds of bias can appear in datasets and trained models?**  
   We will use visual examples and real incidents to help us notice patterns, representation gaps, and underlying assumptions.

2. **Where in the machine learning pipeline does bias enter, and how do design choices amplify or reduce it?**  
   We will connect this to our earlier projects, especially to data preprocessing, model selection, and fine-tuning.

3. **What is the role of fairness and accountability in the lifecycle of a model?**  
   We will learn how concepts like statistical parity, equal opportunity, and disparate impact link to ethical principles of responsibility, transparency, and societal impact.

By the end of this notebook we will have a clear foundation for the rest of Project 12. Later notebooks will build on this foundation by examining:

- how biased representations influence downstream tasks  
- how synthetic media and generative models intensify fairness risks  
- how governance, regulation, and responsible AI frameworks guide real-world decision-making  

> **Key idea:** Bias is not an error in the model. Bias emerges from data, context, history, and decisions. Our task is to understand these influences so that we can design systems that behave reliably and fairly across different groups and situations.


---

In [ ]:
import os
print(os.listdir('NB01_assets/Section01'))

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1148865984", h="3298dbabb7", width=700, height=450) 

### **1. Where Does Bias Enter Machine Learning Pipelines?**

When we trained models in earlier projects, we worked with relatively clean and well-curated datasets. In real-world situations, however, machine learning systems often learn from data that reflects the historical, cultural, and societal patterns of the communities that created it. This means that a large part of what a model learns is not simply “pixel to label,” but a complex mixture of who gets represented, how often, and in what contexts.

In this section, we explore how bias enters the machine learning pipeline by looking at concrete examples from well-known face datasets, image datasets, and recent generative AI systems. The goal here is to develop our intuition: once we can *see* how bias appears in data, we can better understand the downstream effects on model behaviour and fairness.

**1.1 Representation Bias in Face Datasets**

A large portion of fairness research began by analyzing face datasets such as LFWA+, CelebA, COCO, IMDB-WIKI, VGG2, and UTKFace. One of the most influential papers in this space is the FairFace dataset paper (2019). It highlighted that many widely used face datasets were significantly unbalanced across racial groups.

Below is a figure from FairFace [1] illustrating the racial composition of common face datasets:

<![](NB01_assets/Section01/FairfacePaper_racialCompositions_in_face_datasets.png)

This chart reveals that several popular datasets are overwhelmingly composed of images of **White individuals**, with dramatically fewer examples of Black, Latino, East Asian, Southeast Asian, Indian, and Middle Eastern faces. When a model is trained on such a dataset, it forms a skewed internal representation of what “a face” statistically looks like. As a result, recognition accuracy for underrepresented groups tends to be lower—something repeatedly demonstrated in face recognition and fairness studies.

**1.2 How Representation Bias Shows up in Sample Images**

The imbalance we saw in the racial composition plot is not abstract—it becomes very clear when we look at random samples from each dataset.

Below we compare four face datasets (FairFace, UTKFace, LFWA+, CelebA) [1]:

<img src="NB01_assets/Section01/FairfacePaper_randomSamplesFromFaceAttributeDatasets.png" width="600">

We immediately notice visible differences:

- **FairFace** shows a wide distribution of age, skin tone, and ethnicity.  
- **UTKFace** is somewhat mixed but still uneven across groups.  
- **LFWA+ and CelebA** contain mostly White actors, celebrities, and public figures.

When a dataset is dominated by a few demographic groups, a model naturally internalizes those groups as the “default” or “expected” distribution. Everything outside that distribution becomes statistically rare, which leads to poorer performance in downstream tasks.

**1.3 Embedding-Space Evidence: What the Model Actually Learns**

If we go deeper and examine the *representations* learned by a deep network, we see even clearer evidence of dataset bias. The FairFace paper provided a series of t-SNE plots that show how face embeddings cluster by gender and ethnicity [1].

<img src="NB01_assets/Section01/FairfacePaper_tSNE_VisualizationsOfFacesInDatasets.png" width="1000">

Each dot represents a face, and the model’s internal representation groups people with similar features. Notice how:

- Some datasets (like LFWA+) compress nearly all points into one dominant cluster, reflecting their lack of diversity.
- FairFace produces a richer and more evenly distributed embedding space, meaning the model has learned more balanced feature representations.

This reinforces the idea that dataset diversity directly affects the fairness and robustness of trained models.

**1.4 Geographic Bias in Large-Scale Image Datasets**

Bias is not only about faces. Many large-scale image datasets, such as ImageNet and OpenImages, are heavily skewed toward Western countries.

<img src="NB01_assets/Section01/ProportionOfOpenImagesNadImageNetFromEachCountry.jpg" width="900">

We can see that:

- More than **45% of ImageNet** images originate from the United States.  
- Many countries contribute less than 1% of the dataset.  
- This imbalance means the model learns a Western-centric worldview: objects, environments, clothing, food, and cultural artifacts common in non-Western countries are underrepresented or misrepresented.

This geographic skew affects everything from object detection to content moderation and image captioning.


In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1148866052", h="3298dbabb7", width=700, height=450) 

**1.5 Generative Model Bias: How Modern AI Recreates Cultural Stereotypes**

Even when we move to modern generative systems (e.g., diffusion models, large generative transformers), bias persists. These models are trained on massive internet-scale datasets that inherit cultural stereotypes, overgeneralizations, and representational shortcuts.

For example, prompting a generative model with “a Mexican person” often yields nearly identical images following a stereotypical pattern [2]:

<img src="NB01_assets/Section01/RestofWorld_AMexicanPerson.jpg" width="700">

This is not a mistake in the model—it is a learned pattern from the training data. If the internet predominantly represents Mexican people wearing sombreros, the model absorbs that pattern as a statistical truth.

Similarly, prompting “Indonesian food” often yields images that always feature banana leaves, despite Indonesia having a wide culinary diversity [3]:

<img src="NB01_assets/Section01/RestofWorld_IndonesianFoodAlwaysOnBananaLeaves.jpg" width="700">

This illustrates how **models amplify culturally dominant representations**, even when those representations are narrow or stereotypical.

**1.6 Why All These Forms of Bias Matter**

The examples above demonstrate that bias can enter the pipeline long before we begin training, for example:

- **Representation bias**: certain groups appear more frequently or more visibly in the data.  
- **Sampling bias**: some categories dominate the dataset simply because they are easier to collect.  
- **Label bias**: annotations carry human assumptions and cultural norms.  
- **Context bias**: images taken in certain environments or lighting conditions represent only a subset of real-world scenarios.

These imbalances then cascade through the entire ML system:

- Models become less accurate for underrepresented groups.  
- Embedding spaces reflect cultural patterns rather than true visual diversity.  
- Generative systems reproduce stereotypes and amplify societal biases.  
- Downstream decisions—such as identity verification, hiring, or safety monitoring—can become unfair.

Our job in this project is to understand these mechanisms deeply so that we can design models and processes that reduce harm, increase robustness, and uphold fairness as a core design principle rather than an afterthought.

> **Key idea:** Bias does not appear “inside the model.” Bias appears **in the data we give to the model**, in the labels we construct, and in the societal structures that shape both.

**Other Resources for optional visit**: [4] [5]

---

**✅ ALERT: You are now ready to answer the Multiple Choice Questions for this section!**

---

### **2. How Bias Flows Through the Machine Learning Pipeline**

Now that we have seen concrete examples of dataset bias, we step back and ask a more fundamental question: *where* in the machine learning process does this bias enter? The examples in the previous section are symptoms of deeper structural issues. To understand these issues properly, we need to look at the entire ML lifecycle — from data generation to deployment.

Axbom (2023) presents a clear and intuitive diagram (adapted from Suresh & Guttag, 2021) that helps us understand the different points at which bias can appear. We will use this structure to build our intuition.

**2.1 The ML Bias Pipeline**

Below is the high-level pipeline showing **where bias enters before and during model building**, and even after deployment [6]:

<img src="NB01_assets/Section02/axbom_com_machine_learning_biases.png" width="900">

This diagram breaks the lifecycle into two major stages:

1. **Data Generation Biases**  
2. **Model Building and Implementation Biases**

We will walk through each step in narrative form, using real examples from Section 1.

**2.2 Data Generation Biases**

Bias begins long before we train a model. It starts with how the data is produced, collected, labeled, and structured.

**a) Historical Bias**
Historical patterns, inequalities, and stereotypes become embedded in the dataset.  
Even when the dataset is an “accurate representation of the world,” it may still encode discrimination.

For example, in Stable Diffusion generated images, we see disproportionate presentation of skin tome and gender [7]:

  <img src="NB01_assets/Section02/Workers_generated_By_Stable_Diffusion_fromDeepLearning.png" width="800">

Historical bias is not fixed by collecting “more data” — because the underlying societal imbalance is still present.

**b) Representation Bias**
Some groups appear more frequently in the dataset than others.

We saw this clearly in the racial composition chart from FairFace in the last section [1].

<img src="NB01_assets/Section01/FairfacePaper_racialCompositions_in_face_datasets.png" width="600">

This imbalance causes:

- underrepresented groups to be learned poorly  
- embedding spaces to cluster overwhelmingly around dominant groups  
- downstream errors in recognition or detection

Representation bias is one of the most common sources of harm.

**c) Measurement Bias**
This happens when **labels**, **annotations**, or **measurement instruments** are flawed.

Examples:

- Labeling “creditworthiness” using credit scores can encode socioeconomic inequality.  
- Crime prediction models using arrest records inherit policing bias.  
- Gender labels assigned from facial images are oversimplified and culturally dependent.  
- Occupation labels in datasets (e.g., “nurse”, “doctor”) historically skew toward stereotypes.

Measurement bias often goes unnoticed because it happens during labeling, not dataset collection.

**2.3 Model Building and Implementation Biases**

Once we begin model training, different forms of bias appear again — even if the dataset itself were perfectly balanced.

**a) Learning Bias**
During training, model optimization decisions (loss weighting, sampling strategies, architectures) can unintentionally prioritize the majority groups.

For example:
- A classifier may optimize global accuracy by performing very well on the majority group and poorly on minorities.  
- ImageNet-pretrained models may overfocus on texture features present mostly in Western imagery.

Learning bias often explains why even balanced datasets show uneven performance.

**b) Aggregation Bias**
This occurs when a **single model** is used for multiple subpopulations whose distributions differ.

Examples:
- A hiring model trained on “general sentiment” fails for people with different facial expression norms.
- A healthcare model performs worse for groups with different symptom profiles.  
- Embedding models collapse important distinctions (e.g., East Asian vs Southeast Asian features).

Aggregation bias is especially important in deployment settings like healthcare or justice systems.

**c) Evaluation Bias**
We measure the model on benchmark test sets — but if the benchmark is biased, our evaluation is biased too.

For example, Images of dark-skinned women comprise only 7.4% and 4.4% of common benchmark datasets Adience and IJB-A, and thus benchmarking on them fails to discover and penalize underperformance on this part of the population [8].

This means a model could be “state of the art” on paper, yet fail miserably for groups absent from the benchmark.

**d) Deployment Bias**
This occurs when the model is used in a context for which it was **not originally designed**. "This often occurs when a system is built and evaluated as if it were fully autonomous, while in reality, it operates in a complicated sociotechnical system moderated by institutional structures and human decision-makers" [8]. This is also known as the "framing trap" [9]. These tools can lead to harm because of automation or confirmation bias (example: Incorrect advice by an AI-based decision support system could impair the performance of radiologists when reading mammograms).

Deployment bias often emerges from misunderstanding the human-in-the-loop context.

**2.6 The Ecosystem of AI Harms: Beyond Technical Bias**

So far, we have examined how bias enters the machine learning pipeline through data, labeling, learning, evaluation, and deployment. But real-world harm does not come only from technical flaws. It emerges from a much larger **ecosystem of social, organisational, and cultural factors** surrounding the model.

The figure below illustrates this broader landscape of AI harms [10]. It helps us understand that fairness is not just about “fixing datasets.” Fairness depends on who builds the system, who deploys it, who is affected, and what structures surround the technology.

<img src="NB01_assets/Section02/elements_ai_ethics_02.png" width="850">

This diagram separates three layers of concern:

**A. Imagined Harms (science-fiction fear)**

These are the dramatic scenarios often highlighted in media:

- “AI will become conscious and destroy humanity.”  
- “Robots will take over the world.”  
- “Superintelligence will wipe out human life.”

These narratives attract attention, but they often distract us from the **immediate, concrete harms** already affecting people today. Imagined harms are emotionally powerful but do not represent the core fairness challenges we face in real deployments.

**B. Real Harms (already happening today)**

Examples include:

- **Discrimination** — facial recognition that performs poorly on darker-skinned women; hiring systems that downgrade certain groups.  
- **Surveillance** — tracking systems disproportionately deployed in low-income or minority communities.  
- **Stigmatization** — image models reinforcing cultural stereotypes (e.g., “Mexican person” always shown with a sombrero, as seen in Section 1).  
- **Exclusion** — systems that simply do not work for groups missing from the training data.

These harms are not hypothetical. They arise because the technology is shaped by the data and assumptions baked into it. They disproportionately affect people who are underrepresented or historically marginalized.

**C. A System of Interacting Influences**

The diagram highlights six domains:

- **Organisation** — which goals, values, incentives drive the system?  
- **Supervision** — how decisions are monitored and governed?  
- **Environment** — political and cultural forces affecting development?  
- **Human** — who labels data, who collects it, who interprets outputs?  
- **Machine** — model architecture, training, evaluation, deployment?  
- **Society** — who is impacted, who benefits, who is harmed?

Bias does not originate from a single component.  
It emerges from their interaction — often invisibly.

When an ML system fails, it is rarely because “the algorithm is biased.”  
It is because **the ecosystem that produced the algorithm contained structural bias**, which the model absorbed and amplified.

**D. Why This Ecosystem View Matters**

This broader view adds two important insights to our technical pipeline:

1. **Bias is not only inside the data.**  
   It also stems from organisational and societal contexts.

2. **Technical fixes are not enough.**  
   Even a balanced dataset can produce harm if deployed without human oversight, accountability mechanisms, or cultural sensitivity.

This prepares us to think of fairness not as a mathematical constraint but as a **design principle** that involves ethical reasoning, interdisciplinary understanding, and responsible governance.

**Points for Reflection**

- Which parts of this ecosystem connect most strongly to the dataset biases we saw in Section 1?  
- What kinds of harm could occur if a well-trained but poorly governed model is deployed?  
- How does this ecosystem perspective change the way we think about responsibility in AI design?  
- If we were designing a new dataset today, which ecosystem factors would we need to consider beyond pure technical quality?

> **Key idea:** Bias is not a flaw in the model. Bias is a property of the entire socio-technical system in which the model is created, trained, evaluated, and deployed.

---

**✅ ALERT: You are now ready to answer the Multiple Choice Questions for this section!**

---

In [4]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1148866005", h="3298dbabb7", width=700, height=450)

### **3. Real-World Case Studies of Algorithmic Bias**

In this section we slow down and explore a set of real-world examples where algorithmic systems behaved in unexpected, unfair, or harmful ways. These cases help us see that bias is not a theoretical issue — it shows up in deployed systems, influences decision-making, and affects people’s lives. For each case we will combine a figure, a narrative explanation, and a short reflection prompt.

> **Our goal here is not to criticize specific organizations, but to understand why these failures occur and how we, as practitioners, can prevent them.**

**<u>3.1 Case Study 1 — Political Bias in Large Language Models [11]</u>**

<img src="NB01_assets/Section03/PoliticalLeaningOfVariousPretrainedLLMs.png" width="500">

The figure shows how different pretrained language models tend to “sit” on a two-dimensional political map: a social axis (authoritarian ↔ libertarian) and an economic axis (left ↔ right). Each dot corresponds to a model such as BERT, RoBERTa, GPT-2, GPT-3, ChatGPT, or GPT-4. We see that different model families cluster in different regions of this space.

This does not mean that these models “hold political opinions.” Instead, the positions reflect statistical tendencies in their training data: news articles, social media posts, books, and websites that themselves contain ideological patterns. Because large language models learn from human-generated text, they inevitably absorb these patterns and reproduce them in subtle ways.

The key lesson is that models trained on uncurated text data may inherit, and sometimes amplify, the political leanings of the sources they are trained on. If we treat these models as neutral decision-makers, we risk introducing one-sided viewpoints into applications such as news summarization, political chatbots, content moderation, and recommendation systems.

> **Reflection:**  
> If two language models trained on different datasets produce different political tendencies, what does that imply about using them to support tasks like debate generation, policy analysis, or news recommendation?

**<u>3.2 Case Study 2 — COMPAS Recidivism Scoring Bias [12]</u>**

<img src="NB01_assets/Section03/45PercentOfIndividualsInTheDataset_re_offended_59percent_of_them_were_Black.png" width="700">

The COMPAS system is a risk-assessment tool [12] used by several courts in the United States to estimate the likelihood that a person will re-offend. Investigations showed that COMPAS produced systematically higher risk scores for Black defendants than for White defendants, even when we control for prior crimes and other factors.

The figure above summarizes part of this disparity. In the dataset studied, 51% of individuals were Black, yet 59% of those who re-offended and were classified as “high risk” were Black. Further analysis revealed that:

- Black defendants were more likely to be labeled high risk even when they did not re-offend (higher false-positive rate).  
- White defendants were more likely to be labeled low risk even when they did re-offend (higher false-negative rate).

From a fairness perspective, this is deeply problematic. Two people with similar histories receive different risk labels, and therefore potentially different sentences or parole decisions, largely because of race-correlated patterns in the training data. The model is “accurate enough” overall, but the errors are not distributed fairly.

> **Reflection:**  
> When we design a risk prediction system, what kind of fairness should we prioritize: equal overall accuracy, equal false-positive rates, equal false-negative rates, or something else?


In [5]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1148866032", h="3298dbabb7", width=700, height=450) 

**<u>3.3 Case Study 3 — Google Photos and Vision Mislabeling Incidents [13]</u>**

<img src="NB01_assets/Section03/Hand_gun_Monoculer_googlevision_mobile.png" width="400">

In 2015, Google Photos made headlines when it mislabeled photos of Black people as “gorillas.” This incident highlighted how a lack of diverse training data and insufficient auditing can lead to highly offensive and harmful mistakes. Later investigations into Google’s Vision API revealed additional, subtler examples of problematic behavior.

In the example above, we see a dark-skinned hand holding a monocular. The Vision model labels this image with terms including “gun.” When the skin is artificially lightened while keeping the object the same, the model’s predictions shift, and the label “monocular” becomes more prominent. The pixels have changed only in skin tone, yet the system interprets the scene differently.

This is an example of **measurement bias** and **representation bias** interacting:

- The training data appears to have overrepresented associations between darker skin and weapons.  
- The model relies on those associations when making predictions, even when they are inappropriate.

For safety-critical contexts such as surveillance, law enforcement, or content moderation, such misclassifications can have serious consequences, particularly for the groups already facing discrimination.

> **Reflection:**  
> If a vision system is more likely to associate certain objects (for example, “gun”) with specific skin tones, what kinds of harms could occur when the system is deployed at scale?

**<u>3.4 Case Study 4 — Twitter’s Image-Cropping Algorithm Bias [14]</u>**

Twitter (now X) previously used an automated image-cropping algorithm to generate thumbnails. The system tried to crop images around the “most salient” region, often a face. However, users began noticing a pattern: when an image contained multiple faces, the crop frequently favored lighter-skinned faces or particular demographic groups.

**Cropping Examples**

<img src="NB01_assets/Section03/Simpsons_Labradors_imageCropping.png" width="650">

Experiments with cartoon characters and dog photos showed that the cropping system had a systematic preference. Even when faces were equally sized and centrally placed, the algorithm often chose the lighter-skinned character or the lighter-colored dog. This made the bias obvious and easier to communicate.

**Quantitative Evidence**

<img src="NB01_assets/Section03/ImageCroppingPercentage.png" width="700">

The chart above shows the percentage of times the cropping algorithm selected one subject over another in controlled tests. The differences are not random noise; they reveal a consistent skew. The underlying saliency model appears to have been trained on data where certain skin tones, facial features, and image regions were more likely to be considered “important.”

This case illustrates that bias can surface even in seemingly minor UX decisions. A thumbnail might look like a small detail, but it determines which faces and bodies are highlighted in timelines, which in turn affects visibility, representation, and user experience.

> **Reflection:**  
> Why do we think a “neutral” mechanism like saliency scoring still produces biased crops? What assumptions about faces, contrast, or background might be built into the saliency model?

**3.5 What These Cases Teach Us**

Across these four cases we notice some common patterns:

- **Data does not come from nowhere.** It is shaped by history, culture, and power structures. Models that learn from this data inherit those patterns.  
- **Errors are not evenly distributed.** Even when overall accuracy looks acceptable, certain groups may experience much higher error rates or more harmful mistakes.  
- **Design choices matter.** Whether we are predicting recidivism, labeling images, or cropping photos, the way we collect data, define labels, and evaluate performance strongly influences who is treated fairly.

These examples connect back to the bias pipeline and ecosystem that we explored in Section 2. Each case can be traced to specific types of bias:

- representation bias and measurement bias in data collection and labeling,  
- learning and aggregation bias in model training,  
- evaluation and deployment bias when the system is tested and integrated into real workflows.

> **Big picture reflection:**  
> Looking across all four case studies, at which points in the machine learning pipeline could we intervene to prevent or reduce harm? What concrete practices could we adopt (for example, dataset audits, fairness metrics, human-in-the-loop review) to avoid repeating these failures?

---

**✅ ALERT: You are now ready to answer the Multiple Choice Questions for this section!**

---

In [6]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1148866087", h="3298dbabb7", width=700, height=450) 

### **4. Fairness Metrics and Responsible AI Frameworks**

In this section, we take a step back from individual examples and focus on the foundations of fairness in machine learning. Our goal is to understand:

- how fairness is defined,
- how it is measured,
- and how major industry frameworks (Microsoft, Google) guide responsible AI development.

Fairness is not a single number. It is a socio-technical property that emerges from **data**, **models**, **people**, and **institutions**. Metrics help us detect disparities, but human judgement is essential to interpret them.

**4.1 Why Do We Need Fairness Frameworks?**

Fairness problems rarely originate in isolation. They arise when:

- datasets reflect historical or social inequalities,
- model choices amplify or encode those inequalities,
- evaluation overlooks group-level disparities,
- deployment conditions differ from training conditions.

Because of this, many companies emphasize that fairness must be considered holistically. Metrics expose symptoms; principles help us decide what to do next. Here we are discussing about such frameworks from two IT giants, Microsoft and Google.

**4.2 Two Fundamental Types of Harm**

Microsoft frames unfairness in terms of **harm**, especially two main categories [15]:

**1. Harm of allocation**: The system distributes resources or opportunities unevenly. Examples include loan approvals, job recommendations, and medical prioritisation.

**2. Harm of quality-of-service**: The system performs worse for certain groups. Examples include facial recognition failing on darker skin tones or speech recognition missing female voices more often.

These harms help us interpret what fairness metrics tell us.

**4.3 Group Fairness and Disparity Metrics**

Group fairness asks a simple but powerful question:

> Does the model behave differently for different demographic groups?

Microsoft introduces **disparity metrics** that compare performance across groups:

- accuracy disparity  
- error rate disparity  
- precision disparity  
- recall disparity  
- selection rate disparity (e.g., percentage of approved applicants)

A model may be highly accurate overall yet systematically worse for minority groups.  
Disparity metrics help surface these problems.

In [7]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1148866126", h="3298dbabb7", width=700, height=450)

**4.4 Formal Fairness Metrics (Parity Constraints)**

Fairness research defines several mathematical conditions describing how similar predictions should be across groups. These parity constraints appear across Microsoft’s and Google’s work.

**1. Demographic Parity**

A model satisfies demographic parity when positive prediction rates are equal:

$$
P(\hat{Y} = 1 \mid A = a) = P(\hat{Y} = 1 \mid A = b)
$$

Groups should receive positive predictions at similar rates.

**2. Equalized Odds**

A stricter condition requiring equal error behaviour:

$$
P(\hat{Y} = 1 \mid Y = y, A = a) = P(\hat{Y} = 1 \mid Y = y, A = b)
$$

This equalizes:

- true positive rates  
- false positive rates  

across groups.

**3. Equal Opportunity**

A relaxed form of equalized odds:

$$
P(\hat{Y} = 1 \mid Y = 1, A = a) = P(\hat{Y} = 1 \mid Y = 1, A = b)
$$

If someone *actually qualifies*, their chance of a correct prediction should not depend on group membership.

**Comparison Table**

| Fairness Metric | Equalizes | Protects Against | Typical Use |
|-----------------|-----------|------------------|--------------|
| **Demographic Parity** | Selection rates | Allocation harm | Hiring, loan approval |
| **Equal Opportunity** | True positive rates | Denial of qualified individuals | Healthcare triage |
| **Equalized Odds** | TPR and FPR | Unequal error patterns | High-stakes decisions |


**4.5 Fairness Mitigation Techniques**

Microsoft describes two major approaches:

**A. Reduction Algorithms (training-time)**  
Reweight or transform data to satisfy fairness constraints.

**B. Post-Processing (after training)**  
Adjust thresholds separately for groups without retraining the model.

These methods let us balance fairness and accuracy in a controlled way.

**4.6 Responsible AI Principles (Google + Microsoft)**

Google’s 2025 AI Responsibility update emphasizes the role of principles and governance [16]:

- Fairness and inclusion  
- Transparency and explainability  
- Accountability and auditability  
- Safety and robustness  
- Ecological and societal well-being  

Where Microsoft provides the *metrics*, Google provides the *values* and *principles* that shape how we interpret them.

Together they form a complete framework:
**metrics → principles → governance → action**.

**4.7 Connecting Fairness to Project 11 (Transfer Learning)**

The fairness concepts in this section are directly relevant to our P11 ResNet project:

- ImageNet, the backbone we reused, has demographic and geographic skews.
- If our Caltech subset was imbalanced (e.g., many more “airplanes” than “pandas”), feature extraction would reproduce these skews.
- Fine-tuning (NB03) could amplify the imbalance, causing quality-of-service harm.

Even simple image classification benefits from fairness thinking.

**Points for Reflection**

- Which fairness metric best explains the harms shown in Section 3?
- How might an imbalanced Caltech subset affect our P11 ResNet?
- Should we satisfy all fairness constraints simultaneously? Why or why not?
- How could we test whether our P11 backbone inherited representational biases from ImageNet?

---

**✅ ALERT: You are now ready to answer the Multiple Choice Questions for this section!**

---

In [8]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1148866186", h="3298dbabb7", width=700, height=450) 

### **5. Accountability, Governance, and Human Responsibility**

In this final section of NB01 we zoom out from individual bias examples and ask a bigger question:

> **Who is responsible for ensuring that AI systems behave fairly and safely?**

Bias is never just a bug in a model. It emerges from a wider ecosystem of **laws**, **organizational decisions**, and **technical practices**. To understand this, we use two complementary governance frameworks:

- UNESCO’s **Layered Structure of AI Governance** [17]
- The **Hourglass Model of Organizational AI Governance** [19]

Together, they show that fairness is a **shared, multi-layer responsibility** rather than something a single engineer can fix alone.

**<u>5.1 The Layered Structure of AI Governance (UNESCO)</u>**

<img src="NB01_assets/Section05/layerdStructureOfAIgovernance.png" width="420">

UNESCO describes AI governance as three nested layers. Each layer shapes how AI systems are designed, deployed, and monitored.

**1. Environmental Layer – Society, Law, and Norms**

This outer layer includes:

- national and international AI regulations,
- human rights frameworks and ethical principles,
- cultural norms, public debate, and media scrutiny.

It sets the **external boundaries** for what AI systems are allowed to do. For example, the EU AI Act can ban certain “unacceptable risk” applications (such as social scoring) and impose strict conditions on “high-risk” systems.

**2. Organizational Layer – Institutions and Companies**

The organizational layer sits between high-level norms and concrete systems. Here, universities, companies, and public bodies translate external expectations into:

- internal AI policies and value statements,
- risk and impact assessment processes,
- roles and committees (e.g., AI ethics boards),
- documentation and audit practices.

If this translation fails, even technically well-designed models can still be deployed in harmful ways.

**3. AI System Layer – Data, Models, and Pipelines**

This inner layer is where we usually work as ML practitioners:

- data collection and preprocessing,
- model architecture and training,
- evaluation, deployment, and monitoring.

Bias becomes visible here (e.g., skewed accuracy across groups), but it is heavily influenced by decisions made in the outer layers.

> **Key idea:** Fairness is not just a property of a neural network. It is the outcome of interactions between society, organizations, and technical systems.


**<u>5.2 The Hourglass Model of Organizational AI Governance</u>** [18]

<img src="NB01_assets/Section05/hourGlassModel.png" width="800">

The Hourglass Model gives a more operational view of how governance flows through an organization. It looks like an hourglass because **broad external requirements** must pass through a **narrow “translation” layer** before they become **concrete operational practices**.

**Top of the Hourglass – External Requirements**

At the top, we have:

- **Hard law** (regulations, standards),
- **Principles and guidelines** (ethical codes, Responsible AI principles),
- **Stakeholder pressure** (NGOs, customers, affected communities).

These provide broad expectations but are often abstract.

**Middle – Strategic and Value Alignment**

The neck of the hourglass is where abstraction becomes actionable:

- **Strategic alignment** – how AI projects support the organization’s mission while respecting laws and ethics.
- **Value alignment** – how principles such as fairness, transparency, and privacy shape concrete choices.

This is where leadership decisions determine whether ethics is a “nice slide” or a real constraint on projects.

**Bottom – Operational Governance in the AI System Layer**

At the bottom, governance takes the form of everyday practices:

- AI system design and operations,
- algorithm design and documentation,
- risk and impact management,
- data operations and data quality checks,
- development operations (MLOps, deployment),
- transparency and contestation mechanisms,
- compliance, audits, and logging,
- clear **accountability** for who owns which decisions.

> **Interpretation:** The hourglass reminds us that legal and ethical principles only affect real models if organizations actively translate them into processes, tools, and responsibilities.

**<u>5.3 Governance as Shared Responsibility</u>**

Both frameworks emphasise that **no single actor controls fairness**. Instead, responsibilities are distributed:

| **Actor** | **Typical Responsibilities** |
|----------|------------------------------|
| **Developers / Data Scientists** | Curate data, design models, run fairness diagnostics, document limitations, monitor performance. |
| **Organizations / Universities** | Set AI strategy, create governance structures, allocate resources, approve or stop deployments. |
| **Regulators / Governments** | Define legal boundaries, certification schemes, reporting duties, and sanctions. |
| **Users / Society** | Notice and report harms, question unfair use, participate in consultations and public debate. |

If any link in this chain fails, unfair systems can be built and deployed even when the code “works” as intended.

**<u>5.4 Connecting Governance to Our Deep Learning Work (Project 11)</u>**

Our Project 11 experiments (pretrained ResNet, feature extraction, fine-tuning) sit inside the **AI system layer** but are influenced by decisions in the other layers.

Some examples:

- **Dataset curation:**  
  If our Caltech-101 subset over-represents some object classes and under-represents others, we create a *quality-of-service* problem. This is partly a technical choice (which images we include) and partly an organizational issue (how seriously our team treats dataset audits).

- **Transfer learning:**  
  ResNet18 is pretrained on ImageNet, which has its own geographic and demographic biases. When we reuse these weights, we implicitly accept those biases unless we actively test and document them.

- **Fine-tuning and deployment:**  
  If an organization later reuses our fine-tuned model in a high-stakes context (e.g., security surveillance, identity verification) without further checks, responsibility extends beyond us as developers to the organizational and environmental layers.

These examples show how even “teaching projects” raise governance questions once we imagine them in real applications.


**Points for Reflection**

To close this section, we can reflect on the following questions:

- If a biased AI system harms someone, who should be accountable — the individual developer, the organization deploying it, or the regulator that allowed it? Why?

- How does accountability change when we use transfer learning (like in P11), reusing a model that we did not train from scratch?

- Which parts of the layered and hourglass models can we influence directly as ML practitioners, and which require organizational or legal change?

- What governance safeguards would we want in place before deploying our P11 ResNet in a real product?

These questions help us see ourselves not only as model builders but also as participants in a larger governance ecosystem.

---


**Bibliography**

[1] Kärkkäinen, K., & Joo, J. (2019). Fairface: Face attribute dataset for balanced race, gender, and age. arXiv preprint arXiv:1908.04913.</br>
[2] Jun, Y. (2024, April 8). A Brief Overview of Gender Bias in AI. The Gradient. https://thegradient.pub/gender-bias-in-ai/</br>
[3] Turk, V., & Turk, V. (2025, August 4). How AI reduces the world to stereotypes. Rest of World. https://restofworld.org/2023/ai-image-stereotypes/ [Accessed on 27th Nov 2025]</br>
[4] Shankar, S., Halpern, Y., Breck, E., Atwood, J., Wilson, J., & Sculley, D. (2017). No classification without representation: Assessing geodiversity issues in open data sets for the developing world. arXiv preprint arXiv:1711.08536. </br>
[5] Leonardo Nicoletti and Dina Bass, (2023). Humans Are Biased. Generative AI Is Even Worse. Stable Diffusion’s text-to-image model amplifies stereotypes about race and gender — here’s why that matters. The Big TakeTechnology https://www.bloomberg.com/graphics/2023-generative-ai-bias/ [Accessed on 27th Nov 2025]</br>
[6] Per Axbom (2023). Diagram: Bias in Machine Learning. Understand the stages of machine learning where bias can, and often will, contribute to harm. https://axbom.com/bias-in-machine-learning/ [Accessed on 27th Nov 2025]</br>
[7] https://community.deeplearning.ai/t/stable-biases-stable-diffusion-may-amplify-biases-in-its-training-data/385567 [Accessed on 12 Dec 2025]</br>
[8] Buolamwini, J., & Gebru, T. (2018, January). Gender shades: Intersectional accuracy disparities in commercial gender classification. In Conference on fairness, accountability and transparency (pp. 77-91). PMLR.</br>
[9] Selbst, A. D., Boyd, D., Friedler, S. A., Venkatasubramanian, S., & Vertesi, J. (2019, January). Fairness and abstraction in sociotechnical systems. In Proceedings of the conference on fairness, accountability, and transparency (pp. 59-68).</br>
[10] https://axbom.com/aielements/</br>
[11] Scott Bicheno (2023). Inherent bias of AI models confirmed. New research examined the varied biases of different AI large language models, depending on the source material used. https://www.telecoms.com/ai/inherent-bias-of-ai-models-confirmed [Accessed on 27 Nov 2025]</br>
[12] Prathamesh Patalay (2023). COMPAS: Unfair Algorithm? https://medium.com/@lamdaa/compas-unfair-algorithm-812702ed6a6a [Accessed on 27 Nov 2025]</br>
[13] Nicolas Kayser-Bril (2020). Google apologizes after its Vision AI produced racist results. https://algorithmwatch.org/en/google-vision-racism/ [Accessed on 27 Nov 2025]</br>
[14] Jonathan Hui (2021). AI Bias. https://jonathan-hui.medium.com/ai-bias-b85c86bbca90 [Accessed on 27 Nov 2025]</br>
[15] https://learn.microsoft.com/en-us/azure/machine-learning/concept-fairness-ml?view=azureml-api-2 [Accessed on 27 Nov 25]</br>
[16] https://ai.google/static/documents/ai-responsibility-update-published-february-2025.pdf [Accessed on 27 Nov 2025]</br>
[17] https://www.unesco.org/en/artificial-intelligence/recommendation-ethics [Accessed on 27th Nov 25]</br>
[18] https://ai-governance.eu/ai-governance-framework/the-ai-governance-lifecycle/ [Accessed on 27th Nov 25]</br>
[19] Mäntymäki, M., Minkkinen, M., Birkstedt, T., & Viljanen, M. (2022). Putting AI ethics into practice: The hourglass model of organizational AI governance. arXiv preprint arXiv:2206.00335.</br>